In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from datasets import Dataset

# Load the PCL dataset
def load_data():
    df_pcl = pd.read_csv("dontpatronizeme_pcl.tsv", sep="\t", header=None, skiprows=4,
                         names=["par_id", "art_id", "keyword", "country", "text", "label"])

    df_pcl = df_pcl.dropna(subset=['text'])
    
    train_lbl = pd.read_csv("train_semeval_parids-labels.csv")
    dev_lbl = pd.read_csv("dev_semeval_parids-labels.csv")
    
    # Process labels: y=1 if at least one annotator marked PCL
    train_lbl["label_vec"] = train_lbl["label"].apply(ast.literal_eval)
    dev_lbl["label_vec"] = dev_lbl["label"].apply(ast.literal_eval)
    train_lbl["y"] = train_lbl["label_vec"].apply(lambda v: int(sum(v) > 0))
    dev_lbl["y"] = dev_lbl["label_vec"].apply(lambda v: int(sum(v) > 0))
    
    train_df = df_pcl.merge(train_lbl[["par_id", "y"]], on="par_id", how="inner")
    dev_df = df_pcl.merge(dev_lbl[["par_id", "y"]], on="par_id", how="inner")
    
    return train_df, dev_df

train_df, dev_df = load_data()
print(f"Data Loaded. Train size: {len(train_df)}, Dev size: {len(dev_df)}")
dev_df.head()

Data Loaded. Train size: 8375, Dev size: 2093


,par_id,art_id,keyword,country,text,label,y
0,107,@@16900972,homeless,ke,"His present "" chambers "" may be quite humble ,...",3,1
1,149,@@1387882,disabled,us,Krueger recently harnessed that creativity to ...,2,1
2,151,@@19974860,poor-families,in,10:41am - Parents of children who died must ge...,3,1
3,154,@@20663936,disabled,ng,When some people feel causing problem for some...,4,1
4,157,@@21712008,poor-families,ca,We are alarmed to learn of your recently circu...,4,1


In [2]:
def tokenize_fn(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, pos_label=1)
    return {"f1": f1}

# Convert pandas to HuggingFace Dataset format
train_ds = Dataset.from_pandas(train_df[['text', 'y']].rename(columns={'y': 'label'}))
dev_ds = Dataset.from_pandas(dev_df[['text', 'y']].rename(columns={'y': 'label'}))

In [3]:
# Analysis: Count missing values in the text column
print("Missing values in Train:", train_df['text'].isnull().sum())
print("Missing values in Dev:", dev_df['text'].isnull().sum())

# If the count is > 0, see which rows are affected
print("\nRows with missing text in Dev:")
display(dev_df[dev_df['text'].isnull()])

Missing values in Train: 0
Missing values in Dev: 0

Rows with missing text in Dev:


,par_id,art_id,keyword,country,text,label,y


In [4]:
import torch
if torch.backends.mps.is_available():
    print("Apple Silicon GPU (MPS) is available.")
elif torch.cuda.is_available():
    print("NVIDIA GPU (CUDA) is available.")
else:
    print("No GPU found. Training on CPU.")

NVIDIA GPU (CUDA) is available.


# Baseline

In [16]:
import torch
import logging
from transformers import logging as transformers_logging
# 1. Check for CUDA and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
MODEL_NAME_BASE = "roberta-base"
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME_BASE)
# 2. Map datasets (Tokenization happens on CPU/RAM)
tokenized_train_base = train_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)
tokenized_dev_base = dev_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)
# This silences the "Some weights were not initialized..." warnings
transformers_logging.set_verbosity_error()
# 3. Load model and move it to the NVIDIA GPU
model_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_BASE, num_labels=2)
model_base.to(device)
# Optional: Set it back to warning/info after loading so you see training progress
transformers_logging.set_verbosity_info()
training_args_base = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    # --- CUDA SPECIFIC OPTIMIZATIONS ---
    fp16=True,                # Uses half-precision to double speed on NVIDIA GPUs
    per_device_train_batch_size=32,  # GPUs handle larger batches much better than CPUs
    dataloader_num_workers=2,  # Uses multiple CPU cores to feed the GPU data faster
    # ----------------------------------
    logging_steps=50,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)
trainer_base = Trainer(
    model=model_base,
    args=training_args_base,
    train_dataset=tokenized_train_base,
    eval_dataset=tokenized_dev_base,
    compute_metrics=compute_metrics,
)
# 4. Start Training
print("Starting training on CUDA...")
trainer_base.train()

loading configuration file config.json from cache at /home/ubuntu/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.2.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



Using device: cuda


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_baseline/checkpoint-524/model.safetensors
The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 2093
  Batch size = 8
Saving model checkpoint to ./results_baseline/checkpoint-786
Configuration saved in ./results_baseline/checkpoint-786/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_baseline/checkpoint-786/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results_baseline/checkpoint-786 (score: 0.5909090909090909).
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta

TrainOutput(global_step=786, training_loss=0.17368373737383738, metrics={'train_runtime': 67.2894, 'train_samples_per_second': 373.387, 'train_steps_per_second': 11.681, 'total_flos': 1652666316480000.0, 'train_loss': 0.17368373737383738, 'epoch': 3.0})

In [21]:
import torch
import logging
from transformers import logging as transformers_logging

# 1. Check for CUDA and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MODEL_NAME = "roberta-base"
tokenizer_v1 = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Map datasets (Tokenization happens on CPU/RAM)
tokenized_train_v1 = train_ds.map(lambda x: tokenize_fn(x, tokenizer_v1), batched=True)
tokenized_dev_v1   = dev_ds.map(lambda x: tokenize_fn(x, tokenizer_v1),   batched=True)

# This silences the "Some weights were not initialized..." warnings
transformers_logging.set_verbosity_error()

# 3. Load model and move it to the NVIDIA GPU
model_v1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_v1.to(device)
transformers_logging.set_verbosity_info()

training_args_v1 = TrainingArguments(
    output_dir="./results_v1",
    eval_strategy="epoch",
    save_strategy="epoch",
    # --- CUDA SPECIFIC OPTIMIZATIONS ---
    fp16=True,
    per_device_train_batch_size=32,
    dataloader_num_workers=2,
    # --- IMPROVEMENTS OVER BASELINE ---
    logging_steps=50,
    learning_rate=2e-5,
    num_train_epochs=6,            # baseline plateaued at 3, still improving at 6
    weight_decay=0.01,
    warmup_ratio=0.1,              # warmup over first 10% of steps
    lr_scheduler_type="linear",    # decay LR to 0 over training
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_v1 = Trainer(
    model           = model_v1,
    args            = training_args_v1,
    train_dataset   = tokenized_train_v1,
    eval_dataset    = tokenized_dev_v1,
    compute_metrics = compute_metrics,
)

# 4. Start Training
print("Starting v1 training...")
trainer_v1.train()

loading configuration file config.json from cache at /home/ubuntu/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.2.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



Using device: cuda


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_v1/checkpoint-1310/model.safetensors
The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 2093
  Batch size = 8
Saving model checkpoint to ./results_v1/checkpoint-1572
Configuration saved in ./results_v1/checkpoint-1572/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_v1/checkpoint-1572/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results_v1/checkpoint-1572 (score: 0.6).
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.L

TrainOutput(global_step=1572, training_loss=0.12427990722443918, metrics={'train_runtime': 123.2157, 'train_samples_per_second': 407.822, 'train_steps_per_second': 12.758, 'total_flos': 3305332632960000.0, 'train_loss': 0.12427990722443918, 'epoch': 6.0})

# Improved Model

In [19]:
import pandas as pd
from torch import nn

# ── 1. Oversample the minority class ─────────────────────────────────────────
train_pcl    = train_df[train_df["y"] == 1]
train_no_pcl = train_df[train_df["y"] == 0]

oversample_factor = 8   # 794 * 8 ≈ 6352, closer to 7581 No-PCL
train_augmented = pd.concat(
    [train_no_pcl, pd.concat([train_pcl] * oversample_factor, ignore_index=True)],
    ignore_index=True
).sample(frac=1, random_state=42)

print(f"Augmented train size: {len(train_augmented)}")
print(f"PCL rate: {train_augmented['y'].mean():.3f}")

train_ds_aug = Dataset.from_pandas(
    train_augmented[['text', 'y']].rename(columns={'y': 'label'})
)
tokenized_train_aug = train_ds_aug.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)

# ── 2. Class weighted loss ────────────────────────────────────────────────────
n_neg, n_pos = 7581, 794
weight_pos = np.sqrt(n_neg / n_pos)   # ≈ 3.09
class_weights = torch.tensor([1.0, weight_pos], dtype=torch.float).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── 3. Model + training args ──────────────────────────────────────────────────
model_aug = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model_aug.to(device)

training_args_aug = TrainingArguments(
    output_dir="./results_aug",
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    per_device_train_batch_size=32,
    dataloader_num_workers=2,
    logging_steps=50,
    learning_rate=2e-5,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_aug = WeightedTrainer(
    model           = model_aug,
    args            = training_args_aug,
    train_dataset   = tokenized_train_aug,   # ← augmented
    eval_dataset    = tokenized_dev_base,    # ← dev stays the same
    compute_metrics = compute_metrics,
)

print("Starting augmented training...")
trainer_aug.train()

Augmented train size: 13933
PCL rate: 0.456


Map:   0%|          | 0/13933 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /home/ubuntu/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.2.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file model.safetensors from cache at /home/ubuntu/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
PyTorch: setting up devices
The following columns in the Training set don't have a corresponding argument in `Ro

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_aug/checkpoint-2180/model.safetensors
The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 2093
  Batch size = 8
Saving model checkpoint to ./results_aug/checkpoint-2616
Configuration saved in ./results_aug/checkpoint-2616/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results_aug/checkpoint-2616/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results_aug/checkpoint-872 (score: 0.57847533632287).
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.la

TrainOutput(global_step=2616, training_loss=0.07159106006642194, metrics={'train_runtime': 165.9303, 'train_samples_per_second': 503.814, 'train_steps_per_second': 15.766, 'total_flos': 5498889501496320.0, 'train_loss': 0.07159106006642194, 'epoch': 6.0})

In [20]:
from transformers import pipeline
from sklearn.metrics import f1_score
import numpy as np

# Best zero-shot NLI model — pretrained on MNLI, works out of the box
nli_classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-roberta-base",
    device=0   # use GPU; set to -1 if CPU
)

hypothesis = "This text is patronising or condescending towards a vulnerable group."

def predict_pcl_nli(texts, batch_size=32):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = nli_classifier(batch, candidate_labels=["patronising", "not patronising"])
        for r in results:
            # If top label is "patronising" → predict 1
            preds.append(1 if r["labels"][0] == "patronising" else 0)
        if i % 200 == 0:
            print(f"  {i}/{len(texts)} done...")
    return np.array(preds)

# Run on dev set
dev_texts  = dev_df["text"].tolist()
dev_labels = dev_df["y"].tolist()

print("Running NLI classifier on dev set...")
dev_preds_nli = predict_pcl_nli(dev_texts)

f1 = f1_score(dev_labels, dev_preds_nli, pos_label=1)
print(f"\nNLI zero-shot Dev F1: {f1:.4f}")

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

loading configuration file config.json from cache at /home/ubuntu/.cache/huggingface/hub/models--cross-encoder--nli-roberta-base/snapshots/1be0567456f0543475805e758725f151f283705a/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "contradiction",
    "1": "entailment",
    "2": "neutral"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "contradiction": 0,
    "entailment": 1,
    "neutral": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transfor

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /home/ubuntu/.cache/huggingface/hub/models--cross-encoder--nli-roberta-base/snapshots/1be0567456f0543475805e758725f151f283705a/model.safetensors
Since the `dtype` attribute can't be found in model's config object, will use dtype={dtype} as derived from model's weights


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Disabling tokenizer parallelism, we're using DataLoader multithreading already


Running NLI classifier on dev set...
  0/2093 done...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  800/2093 done...
  1600/2093 done...

NLI zero-shot Dev F1: 0.0984


# Compare Models

In [ ]:
def get_report(trainer, tokenized_data, title):
    print(f"\n--- {title} Results ---")
    preds = trainer.predict(tokenized_data)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    print(classification_report(preds.label_ids, pred_labels, target_names=["No PCL", "PCL"]))

get_report(trainer_base, tokenized_dev_base, "RoBERTa Baseline")
get_report(trainer_imp, tokenized_dev_imp, "DeBERTa Improved")